# 03 — Evaluating the feedback intelligence platform

> **Applied Labs** · **Domain**: PE · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/09_feedback_intelligence/03_evaluate.ipynb)

**Prerequisites**:
- [02 — Building the feedback intelligence platform](./02_build.ipynb)
- Familiarity with precision, recall, F1, and confusion matrices

**What you will learn**:
- How to evaluate aspect extraction with precision, recall, and F1 at the aspect level
- Sentiment classification accuracy and the confusion matrix for per-aspect sentiment
- Trend detection evaluation: precision, recall, and time-to-detection metrics
- LLM-as-judge for evaluating summary quality on accuracy, completeness, and actionability
- Cost modelling and ROI analysis for feedback intelligence at different scales

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0" "scikit-learn>=1.2.0"

import os, json, re, textwrap
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import defaultdict, Counter
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

np.random.seed(42)

# --- OpenAI setup ---
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = None
if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM-as-judge cells will use mock scores")

print("Setup complete ✓")

## Section 1 — Evaluation framework

We evaluate the feedback intelligence platform across **five dimensions**,
each targeting a different pipeline stage:

| Stage | Metric | Gold standard |
|---|---|---|
| Aspect extraction | Precision, Recall, F1 | 30 hand-labelled reviews |
| Sentiment scoring | Accuracy, confusion matrix | Per-aspect gold labels |
| Trend detection | Precision, Recall, time-to-detection | Known planted anomalies |
| Summary quality | LLM-as-judge (4 criteria) | Human evaluation rubric |
| Cost & throughput | $/1000 reviews, time/review | Baseline comparisons |

Each metric answers a specific business question:
- **Extraction accuracy**: Are we finding the right issues?
- **Sentiment accuracy**: Are we correctly gauging customer feelings?
- **Trend detection**: Are we catching problems early enough?
- **Summary quality**: Are the briefs trustworthy for decision-making?
- **Cost**: Does the ROI justify deployment?

In [ ]:
# --- Evaluation framework dataclass ---

@dataclass
class EvalResult:
    """Result from a single evaluation."""
    stage: str
    metric: str
    value: float
    details: str = ""

class EvalFramework:
    """Collect and report evaluation results."""

    def __init__(self):
        self.results: List[EvalResult] = []

    def add(self, stage: str, metric: str, value: float, details: str = ""):
        self.results.append(EvalResult(stage, metric, value, details))

    def report(self) -> pd.DataFrame:
        """Generate evaluation report."""
        df = pd.DataFrame([vars(r) for r in self.results])
        return df

    def summary(self):
        """Print human-readable summary."""
        print("\n" + "=" * 60)
        print("  EVALUATION SUMMARY")
        print("=" * 60)
        for r in self.results:
            indicator = "✓" if r.value >= 0.7 else ("△" if r.value >= 0.5 else "✗")
            print(f"  {indicator} {r.stage:<22} {r.metric:<25} {r.value:.3f}")
            if r.details:
                print(f"    → {r.details}")
        print("=" * 60)

eval_fw = EvalFramework()
print("✓ Evaluation framework initialised")

## Section 2 — Aspect extraction accuracy

We create **30 gold-labelled reviews** with manually annotated aspects.
For each review, we compare the system's extracted aspects against ground
truth and compute **precision** (no hallucinated aspects), **recall** (no
missed aspects), and **F1**.

**Error categories**:
- *Missed aspects*: real issues the system failed to detect
- *Hallucinated aspects*: aspects the system invented that aren't in the text
- *Miscategorised aspects*: correct detection but wrong category label

In [ ]:
# --- Gold-labelled test set (30 reviews) ---
GOLD_LABELS: List[Dict] = [
    {"text": "Love the product quality but support is terrible", "aspects": ["product_quality", "customer_support"], "sentiments": ["positive", "negative"]},
    {"text": "Login keeps timing out since Tuesday", "aspects": ["login"], "sentiments": ["negative"]},
    {"text": "Dashboard looks great, very intuitive design", "aspects": ["ui_ux"], "sentiments": ["positive"]},
    {"text": "Way too expensive for small teams", "aspects": ["pricing"], "sentiments": ["negative"]},
    {"text": "Setup was a breeze, got running in 5 minutes", "aspects": ["onboarding"], "sentiments": ["positive"]},
    {"text": "Billing is confusing and I got charged twice", "aspects": ["billing"], "sentiments": ["negative"]},
    {"text": "App crashes every time I open the analytics page", "aspects": ["performance"], "sentiments": ["negative"]},
    {"text": "Customer support responded within an hour, very helpful", "aspects": ["customer_support"], "sentiments": ["positive"]},
    {"text": "The interface is beautiful but loading times are awful", "aspects": ["ui_ux", "performance"], "sentiments": ["positive", "negative"]},
    {"text": "Great value for money, best tool we've used", "aspects": ["pricing", "product_quality"], "sentiments": ["positive", "positive"]},
    {"text": "Login broken on mobile, works fine on desktop", "aspects": ["login"], "sentiments": ["negative"]},
    {"text": "Onboarding docs are outdated and confusing", "aspects": ["onboarding"], "sentiments": ["negative"]},
    {"text": "Performance has improved a lot since last month", "aspects": ["performance"], "sentiments": ["positive"]},
    {"text": "Billing page needs a complete redesign", "aspects": ["billing", "ui_ux"], "sentiments": ["negative", "negative"]},
    {"text": "Excellent product, terrible documentation", "aspects": ["product_quality", "onboarding"], "sentiments": ["positive", "negative"]},
    {"text": "Support agent was rude and unhelpful", "aspects": ["customer_support"], "sentiments": ["negative"]},
    {"text": "The new dashboard update is fantastic", "aspects": ["ui_ux"], "sentiments": ["positive"]},
    {"text": "Keeps logging me out randomly", "aspects": ["login"], "sentiments": ["negative"]},
    {"text": "Invoice format changed without notice, very confusing", "aspects": ["billing"], "sentiments": ["negative"]},
    {"text": "Fast, reliable, and easy to use", "aspects": ["performance", "product_quality"], "sentiments": ["positive", "positive"]},
    {"text": "Pricing tiers don't make sense for our usage pattern", "aspects": ["pricing"], "sentiments": ["negative"]},
    {"text": "Love the UI but hate the pricing", "aspects": ["ui_ux", "pricing"], "sentiments": ["positive", "negative"]},
    {"text": "Authentication flow is smooth and secure", "aspects": ["login"], "sentiments": ["positive"]},
    {"text": "Product is okay, nothing special", "aspects": ["product_quality"], "sentiments": ["neutral"]},
    {"text": "Support took 3 days to respond to a critical issue", "aspects": ["customer_support"], "sentiments": ["negative"]},
    {"text": "Crash on export, lost all my data", "aspects": ["performance", "product_quality"], "sentiments": ["negative", "negative"]},
    {"text": "Setup wizard is excellent, guided me through everything", "aspects": ["onboarding"], "sentiments": ["positive"]},
    {"text": "Monthly billing but no way to switch to annual", "aspects": ["billing", "pricing"], "sentiments": ["negative", "negative"]},
    {"text": "The API is blazing fast", "aspects": ["performance"], "sentiments": ["positive"]},
    {"text": "Good product overall but could use better support", "aspects": ["product_quality", "customer_support"], "sentiments": ["positive", "negative"]},
]

# --- Aspect extractor (matching the build notebook) ---
def extract_aspects(text: str) -> List[str]:
    """Extract aspect categories from text (keyword-based)."""
    text_lower = text.lower()
    aspect_signals = {
        "login": ["login", "log in", "sign in", "timeout", "authentication", "logging"],
        "billing": ["billing", "charged", "invoice", "refund", "payment", "billing page"],
        "ui_ux": ["interface", "ui", "ux", "dashboard", "design", "navigation", "looks"],
        "customer_support": ["support", "help", "agent", "response", "responded"],
        "performance": ["performance", "loading", "slow", "fast", "crash", "speed", "blazing", "reliable"],
        "pricing": ["pricing", "cost", "expensive", "price", "value", "money", "tiers"],
        "product_quality": ["product", "quality", "build", "tool", "overall"],
        "onboarding": ["setup", "onboarding", "getting started", "docs", "documentation", "wizard"],
    }
    found = []
    for aspect, keywords in aspect_signals.items():
        if any(kw in text_lower for kw in keywords):
            found.append(aspect)
    return found if found else ["general"]

# --- Evaluate ---
all_true_aspects: List[Set[str]] = []
all_pred_aspects: List[Set[str]] = []
missed_examples: List[Tuple[str, str]] = []
hallucinated_examples: List[Tuple[str, str]] = []

for gold in GOLD_LABELS:
    true_set = set(gold["aspects"])
    pred_set = set(extract_aspects(gold["text"]))

    all_true_aspects.append(true_set)
    all_pred_aspects.append(pred_set)

    missed = true_set - pred_set
    hallucinated = pred_set - true_set
    for m in missed:
        missed_examples.append((gold["text"][:50], m))
    for h in hallucinated:
        hallucinated_examples.append((gold["text"][:50], h))

# Compute micro-averaged P/R/F1
ALL_ASPECTS = sorted(set(a for s in all_true_aspects for a in s) |
                     set(a for s in all_pred_aspects for a in s))

tp = fp = fn = 0
for true_set, pred_set in zip(all_true_aspects, all_pred_aspects):
    tp += len(true_set & pred_set)
    fp += len(pred_set - true_set)
    fn += len(true_set - pred_set)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

eval_fw.add("aspect_extraction", "precision", precision, f"{tp} TP, {fp} FP")
eval_fw.add("aspect_extraction", "recall", recall, f"{tp} TP, {fn} FN")
eval_fw.add("aspect_extraction", "f1", f1)

print(f"Aspect Extraction Evaluation (30 reviews):\n")
print(f"  Precision: {precision:.3f} ({tp} correct, {fp} hallucinated)")
print(f"  Recall:    {recall:.3f} ({tp} found, {fn} missed)")
print(f"  F1 Score:  {f1:.3f}")

if missed_examples:
    print(f"\nMissed aspects ({len(missed_examples)} total, showing top 5):")
    for text, asp in missed_examples[:5]:
        print(f"  ✗ "{text}..." → missed '{asp}'")

if hallucinated_examples:
    print(f"\nHallucinated aspects ({len(hallucinated_examples)} total, showing top 5):")
    for text, asp in hallucinated_examples[:5]:
        print(f"  ⚠ "{text}..." → hallucinated '{asp}'")

print(f"\n✓ Aspect extraction evaluation complete")

## Section 3 — Sentiment accuracy

For each aspect in our gold-labelled set, we evaluate whether the system
correctly identifies the sentiment (positive, negative, neutral). We compute
a **confusion matrix** and identify which aspect-sentiment pairs are hardest.

In [ ]:
# --- Sentiment extractor ---
def extract_sentiment(text: str) -> str:
    """Simple keyword-based sentiment classifier."""
    positive_kw = {"love", "great", "excellent", "amazing", "fantastic", "smooth",
                   "intuitive", "helpful", "beautiful", "fast", "blazing", "best",
                   "good", "improved", "breeze", "excellent", "guided"}
    negative_kw = {"terrible", "awful", "broken", "confusing", "expensive", "crashes",
                   "slow", "poor", "rude", "unhelpful", "outdated", "lost", "hate",
                   "critical", "timing out", "logging me out"}
    words = set(text.lower().split())
    pos = len(words & positive_kw)
    neg = len(words & negative_kw)
    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    return "neutral"

# --- Evaluate per-aspect sentiment ---
y_true: List[str] = []
y_pred: List[str] = []
errors_by_aspect: Dict[str, List[Tuple[str, str, str]]] = defaultdict(list)

for gold in GOLD_LABELS:
    # For multi-aspect reviews, evaluate each aspect separately
    for i, (asp, true_sent) in enumerate(zip(gold["aspects"], gold["sentiments"])):
        pred_sent = extract_sentiment(gold["text"])
        y_true.append(true_sent)
        y_pred.append(pred_sent)
        if pred_sent != true_sent:
            errors_by_aspect[asp].append((gold["text"][:50], true_sent, pred_sent))

# Compute metrics
labels = ["positive", "negative", "neutral"]
accuracy = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)
eval_fw.add("sentiment", "accuracy", accuracy, f"{len(y_true)} aspect-sentiments evaluated")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels],
                      columns=[f"pred_{l}" for l in labels])

print(f"Sentiment Evaluation ({len(y_true)} aspect-sentiment pairs):\n")
print(f"  Overall accuracy: {accuracy:.3f} ({sum(1 for t, p in zip(y_true, y_pred) if t == p)}/{len(y_true)})")

print(f"\nConfusion matrix:")
print(cm_df.to_string())

# Per-class metrics
print(f"\nPer-class metrics:")
for label in labels:
    true_mask = [t == label for t in y_true]
    pred_mask = [p == label for p in y_pred]
    tp = sum(1 for t, p in zip(true_mask, pred_mask) if t and p)
    fp = sum(1 for t, p in zip(true_mask, pred_mask) if not t and p)
    fn = sum(1 for t, p in zip(true_mask, pred_mask) if t and not p)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    print(f"  {label:>10}: P={p:.3f}  R={r:.3f}  F1={f:.3f}")

# Hardest aspects
print(f"\nMost error-prone aspects:")
for asp, errs in sorted(errors_by_aspect.items(), key=lambda x: len(x[1]), reverse=True)[:5]:
    print(f"  {asp}: {len(errs)} errors")
    for text, true_s, pred_s in errs[:2]:
        print(f"    "{text}..." — true={true_s}, pred={pred_s}")

# Visualise
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([f"pred_{l}" for l in labels])
ax.set_yticklabels([f"true_{l}" for l in labels])
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
ax.set_title("Sentiment Confusion Matrix")
plt.colorbar(im)
plt.tight_layout()
plt.show()

print(f"\n✓ Sentiment evaluation complete")

## Section 4 — Trend detection evaluation

We evaluate against **known anomalies** planted in the synthetic dataset:
1. **Login bug** — emerging from day 15, accelerating through day 30
2. **Billing spike** — days 27–30 (month-end seasonal)

Metrics:
- **Detection precision**: of all flagged anomalies, how many are real?
- **Detection recall**: of all real anomalies, how many were flagged?
- **Time-to-detection**: how many days after the anomaly starts is it first flagged?

In [ ]:
# --- Known ground truth anomalies ---
GROUND_TRUTH_ANOMALIES = {
    "login": {"start_day": 15, "end_day": 30, "type": "emerging_issue"},
    "billing": {"start_day": 27, "end_day": 30, "type": "seasonal_spike"},
}

# --- Rebuild trend data (matching build notebook) ---
np.random.seed(42)
n_days = 30
date_range = pd.date_range("2025-01-01", periods=n_days, freq="D")
aspects_data = {
    "login": np.random.poisson(4, n_days).astype(float),
    "billing": np.random.poisson(3, n_days).astype(float),
    "ui_ux": np.random.poisson(5, n_days).astype(float),
    "performance": np.random.poisson(4, n_days).astype(float),
    "customer_support": np.random.poisson(4, n_days).astype(float),
    "product_quality": np.random.poisson(3, n_days).astype(float),
}
aspects_data["login"][15:] += np.linspace(2, 15, n_days - 15)
aspects_data["billing"][27:] += np.array([8, 10, 6])[:n_days - 27]

df_ts = pd.DataFrame(aspects_data, index=date_range)

# --- Run detection at multiple thresholds ---
thresholds = [1.5, 2.0, 2.5, 3.0]
threshold_results: List[Dict] = []

for z_thresh in thresholds:
    tp = fp = fn = 0
    detections: Dict[str, Optional[int]] = {}

    for aspect in df_ts.columns:
        series = df_ts[aspect].astype(float)
        ma = series.rolling(7, min_periods=3).mean()
        std = series.rolling(7, min_periods=3).std().replace(0, 1)
        z = (series - ma) / std
        flagged_days = set(np.where(z > z_thresh)[0])

        if aspect in GROUND_TRUTH_ANOMALIES:
            gt = GROUND_TRUTH_ANOMALIES[aspect]
            true_days = set(range(gt["start_day"], gt["end_day"]))
            tp_days = flagged_days & true_days
            fp_days = flagged_days - true_days
            fn_days = true_days - flagged_days
            tp += len(tp_days)
            fp += len(fp_days)
            fn += len(fn_days)

            # Time to detection
            if tp_days:
                first_detection = min(tp_days)
                ttd = first_detection - gt["start_day"]
                detections[aspect] = ttd
            else:
                detections[aspect] = None
        else:
            fp += len(flagged_days)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_val = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    threshold_results.append({
        "threshold": z_thresh, "precision": precision, "recall": recall,
        "f1": f1_val, "tp": tp, "fp": fp, "fn": fn,
        "detections": detections
    })

# Best F1 threshold
best = max(threshold_results, key=lambda r: r["f1"])
eval_fw.add("trend_detection", "precision", best["precision"], f"z-threshold={best['threshold']}")
eval_fw.add("trend_detection", "recall", best["recall"])
eval_fw.add("trend_detection", "f1", best["f1"])

print("Trend Detection Evaluation:\n")
print(f"  {'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'TP':>5} {'FP':>5} {'FN':>5}")
print("  " + "─" * 55)
for r in threshold_results:
    marker = " ←" if r == best else ""
    print(f"  {r['threshold']:>10.1f} {r['precision']:>10.3f} {r['recall']:>10.3f} "
          f"{r['f1']:>10.3f} {r['tp']:>5} {r['fp']:>5} {r['fn']:>5}{marker}")

print(f"\nTime-to-detection (best threshold z={best['threshold']}):")
for aspect, ttd in best["detections"].items():
    if ttd is not None:
        print(f"  {aspect}: detected {ttd} days after anomaly start")
    else:
        print(f"  {aspect}: NOT DETECTED ✗")

# Visualise P/R/F1 vs threshold
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([r["threshold"] for r in threshold_results],
        [r["precision"] for r in threshold_results], "o-", label="Precision")
ax.plot([r["threshold"] for r in threshold_results],
        [r["recall"] for r in threshold_results], "s-", label="Recall")
ax.plot([r["threshold"] for r in threshold_results],
        [r["f1"] for r in threshold_results], "^-", label="F1")
ax.set_xlabel("Z-score threshold")
ax.set_ylabel("Score")
ax.set_title("Trend Detection — Precision/Recall vs Z-score Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Trend detection evaluation complete")

## Section 5 — Summary quality

We evaluate executive summaries using **LLM-as-judge** on four criteria:
1. **Accurate** — Are the numbers and claims factually correct?
2. **Comprehensive** — Does it cover all important issues?
3. **Actionable** — Are the recommendations specific and implementable?
4. **Well-prioritised** — Are the most important issues ranked first?

Each criterion is scored 1–5. We evaluate 5 sample summaries.

In [ ]:
# --- Sample summaries to evaluate ---
SAMPLE_SUMMARIES = [
    {
        "context": "200 reviews, login bugs spiking (150 mentions), billing complaints (45), UI positive (80)",
        "summary": "Over the past 30 days, login failures have become the dominant issue with 150 mentions, "
                   "a 200% increase since day 15. Billing confusion affects 45 users, primarily at month-end. "
                   "UI sentiment remains positive (80 mentions, 90% positive). Recommend: (1) hotfix for login "
                   "timeout bug, (2) billing page redesign, (3) continue UI investment.",
    },
    {
        "context": "200 reviews, login bugs spiking (150 mentions), billing complaints (45), UI positive (80)",
        "summary": "Customers are having some issues. Login seems problematic. Billing could be improved. "
                   "The UI is generally well-received. We should look into things.",
    },
    {
        "context": "200 reviews, login bugs spiking (150 mentions), billing complaints (45), UI positive (80)",
        "summary": "Login failures: 150 mentions, +200% trend. This is a P0 that requires immediate "
                   "engineering investigation — likely related to the Jan 15 deployment. Billing: 45 mentions, "
                   "seasonal month-end pattern. Recommend auto-generated invoice summary. UI: 80 positive "
                   "mentions — maintain current design direction.",
    },
    {
        "context": "200 reviews, performance good, support complaints rising, pricing neutral",
        "summary": "Performance is rated positively across 60 mentions. Support complaints have risen 40% "
                   "this month with average response time cited as the main concern. Pricing sentiment is "
                   "neutral. Actions: (1) hire 2 additional support agents, (2) implement SLA monitoring, "
                   "(3) A/B test pricing page clarity.",
    },
    {
        "context": "200 reviews, all aspects stable, no anomalies",
        "summary": "All feedback metrics are within normal ranges this period. No emerging issues detected. "
                   "Product quality leads positive sentiment (35 mentions). Recommend: maintain current "
                   "trajectory, schedule quarterly deep-dive for next month.",
    },
]

JUDGE_PROMPT = """Rate this executive summary on a scale of 1-5 for each criterion.

CONTEXT: {context}
SUMMARY: {summary}

Rate each criterion (1=poor, 5=excellent):
1. ACCURATE — Are numbers and claims factually correct?
2. COMPREHENSIVE — Does it cover all important issues?
3. ACTIONABLE — Are recommendations specific and implementable?
4. WELL_PRIORITISED — Are the most important issues ranked first?

Return JSON: {{"accurate": N, "comprehensive": N, "actionable": N, "well_prioritised": N, "reasoning": "..."}}
"""

def judge_summary(context: str, summary: str) -> Dict[str, float]:
    """Score summary using LLM-as-judge or heuristic fallback."""
    if client:
        try:
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": JUDGE_PROMPT.format(
                    context=context, summary=summary)}],
                temperature=0.1, max_tokens=300,
            )
            content = resp.choices[0].message.content.strip()
            match = re.search(r"\{.*\}", content, re.DOTALL)
            if match:
                scores = json.loads(match.group())
                return {k: float(v) for k, v in scores.items() if k != "reasoning"}
        except Exception:
            pass

    # --- Heuristic fallback ---
    score: Dict[str, float] = {}
    words = summary.lower().split()
    has_numbers = bool(re.search(r"\d+", summary))
    has_actions = any(w in summary.lower() for w in ["recommend", "action", "implement", "hire", "fix", "hotfix"])
    has_specifics = len(words) > 30
    has_priority = any(w in summary.lower() for w in ["p0", "first", "immediate", "urgent", "primary"])

    score["accurate"] = 4.0 if has_numbers else 2.0
    score["comprehensive"] = 4.0 if has_specifics else 2.0
    score["actionable"] = 4.5 if has_actions else 1.5
    score["well_prioritised"] = 4.0 if has_priority else 2.0
    return score

# --- Evaluate all summaries ---
print("Summary Quality Evaluation (5 summaries):\n")
all_scores: List[Dict[str, float]] = []

for i, sample in enumerate(SAMPLE_SUMMARIES):
    scores = judge_summary(sample["context"], sample["summary"])
    all_scores.append(scores)
    avg = np.mean(list(scores.values()))
    print(f"  Summary {i+1}: avg={avg:.1f}/5.0")
    for criterion, score in scores.items():
        bar = "█" * int(score) + "░" * (5 - int(score))
        print(f"    {criterion:<20} {score:.1f}/5  {bar}")
    print()

# Aggregate scores
criteria = ["accurate", "comprehensive", "actionable", "well_prioritised"]
avg_scores = {c: np.mean([s.get(c, 0) for s in all_scores]) for c in criteria}
overall_avg = np.mean(list(avg_scores.values()))
eval_fw.add("summary_quality", "overall_avg", overall_avg / 5.0,
            f"Avg {overall_avg:.1f}/5.0 across {len(SAMPLE_SUMMARIES)} summaries")

print(f"Aggregate scores across {len(SAMPLE_SUMMARIES)} summaries:")
for c, avg in avg_scores.items():
    print(f"  {c:<20} {avg:.2f}/5.0")
print(f"  {'OVERALL':<20} {overall_avg:.2f}/5.0")
print(f"\n✓ Summary quality evaluation complete")

## Section 6 — Cost and throughput

We model the costs of running the feedback intelligence pipeline at different
scales and compare against manual analyst costs.

Key factors:
- **LLM cost**: token usage per review (extraction + summarisation)
- **Compute cost**: trend detection and alerting (CPU-only, negligible)
- **Manual baseline**: $45/hour analyst processing ~8 reviews/hour

In [ ]:
# --- Cost model ---

@dataclass
class CostModel:
    """Cost per 1000 reviews at different pipeline stages."""
    extraction_cost_per_1k: float = 0.0    # LLM extraction
    summary_cost_per_1k: float = 0.0       # LLM summarisation
    compute_cost_per_1k: float = 0.0       # CPU for trend/alert
    total_cost_per_1k: float = 0.0
    time_per_review_sec: float = 0.0       # average processing time

# --- Model parameters ---
# GPT-4o-mini pricing: $0.15/1M input, $0.60/1M output
AVG_INPUT_TOKENS = 200    # per review (extraction prompt + text)
AVG_OUTPUT_TOKENS = 150   # per review (structured JSON response)
SUMMARY_INPUT_TOKENS = 2000   # per batch (aspect data context)
SUMMARY_OUTPUT_TOKENS = 300   # per batch

INPUT_COST_PER_TOKEN = 0.15 / 1_000_000
OUTPUT_COST_PER_TOKEN = 0.60 / 1_000_000

# Compute costs
extraction_cost = (AVG_INPUT_TOKENS * INPUT_COST_PER_TOKEN +
                   AVG_OUTPUT_TOKENS * OUTPUT_COST_PER_TOKEN) * 1000
summary_cost = (SUMMARY_INPUT_TOKENS * INPUT_COST_PER_TOKEN +
                SUMMARY_OUTPUT_TOKENS * OUTPUT_COST_PER_TOKEN)  # per batch, amortised
compute_cost = 0.05  # negligible CPU cost per 1K reviews

llm_model = CostModel(
    extraction_cost_per_1k=round(extraction_cost, 2),
    summary_cost_per_1k=round(summary_cost, 4),
    compute_cost_per_1k=compute_cost,
    total_cost_per_1k=round(extraction_cost + summary_cost + compute_cost, 2),
    time_per_review_sec=0.8,
)

manual_cost_per_1k = (1000 / 8) * 45  # 8 reviews/hr at $45/hr
legacy_nlp_cost_per_1k = 15.0

# --- Scale analysis ---
volumes = [100, 1_000, 5_000, 10_000, 50_000, 100_000]
print("Cost Analysis — LLM vs Manual vs Legacy NLP\n")
print(f"  {'Volume':>10} {'LLM ($)':>10} {'Legacy ($)':>10} {'Manual ($)':>12} {'LLM Savings':>12}")
print("  " + "─" * 58)

for vol in volumes:
    llm_cost = llm_model.total_cost_per_1k * vol / 1000
    manual_cost = manual_cost_per_1k * vol / 1000
    legacy_cost = legacy_nlp_cost_per_1k * vol / 1000
    savings_pct = (1 - llm_cost / manual_cost) * 100
    print(f"  {vol:>10,} ${llm_cost:>9,.2f} ${legacy_cost:>9,.2f} ${manual_cost:>11,.2f} {savings_pct:>11.1f}%")

# --- ROI analysis ---
print(f"\n\nROI Analysis (mid-market SaaS: 10,000 reviews/month):\n")
monthly_volume = 10_000
llm_monthly = llm_model.total_cost_per_1k * monthly_volume / 1000
manual_monthly = manual_cost_per_1k * monthly_volume / 1000
savings_monthly = manual_monthly - llm_monthly
annual_savings = savings_monthly * 12

print(f"  LLM pipeline cost:    ${llm_monthly:>10,.2f}/month")
print(f"  Manual analyst cost:  ${manual_monthly:>10,.2f}/month")
print(f"  Monthly savings:      ${savings_monthly:>10,.2f}/month")
print(f"  Annual savings:       ${annual_savings:>10,.2f}/year")
print(f"  Break-even:           < 1 month")

eval_fw.add("cost", "llm_cost_per_1k", llm_model.total_cost_per_1k, "GPT-4o-mini pricing")
eval_fw.add("cost", "savings_vs_manual", savings_pct / 100,
            f"{savings_pct:.1f}% cheaper than manual at 100K volume")

# --- Throughput ---
print(f"\n\nThroughput Analysis:\n")
print(f"  LLM pipeline:    {1/llm_model.time_per_review_sec:.0f} reviews/sec "
      f"({llm_model.time_per_review_sec:.1f}s per review)")
print(f"  Manual analyst:  {8/3600:.4f} reviews/sec "
      f"({3600/8:.0f}s per review)")
speedup = (3600 / 8) / llm_model.time_per_review_sec
print(f"  Speedup:         {speedup:,.0f}×")

eval_fw.add("throughput", "reviews_per_sec", 1/llm_model.time_per_review_sec)
eval_fw.add("throughput", "speedup_vs_manual", min(speedup / 1000, 1.0),
            f"{speedup:,.0f}× faster than manual")

# --- Final evaluation summary ---
eval_fw.summary()

# Visualise cost comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cost at scale
ax1.plot(volumes, [llm_model.total_cost_per_1k * v / 1000 for v in volumes],
         "o-", label="LLM Pipeline", color="tab:blue")
ax1.plot(volumes, [legacy_nlp_cost_per_1k * v / 1000 for v in volumes],
         "s-", label="Legacy NLP", color="tab:orange")
ax1.plot(volumes, [manual_cost_per_1k * v / 1000 for v in volumes],
         "^-", label="Manual Analyst", color="tab:red")
ax1.set_xlabel("Monthly reviews")
ax1.set_ylabel("Monthly cost ($)")
ax1.set_title("Cost at Scale")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Eval radar chart (simplified as bar)
metrics = ["Extraction\nF1", "Sentiment\nAccuracy", "Trend\nF1",
           "Summary\nQuality", "Cost\nEfficiency"]
values = [
    eval_fw.results[2].value if len(eval_fw.results) > 2 else 0,   # extraction F1
    eval_fw.results[3].value if len(eval_fw.results) > 3 else 0,   # sentiment acc
    eval_fw.results[6].value if len(eval_fw.results) > 6 else 0,   # trend F1
    eval_fw.results[7].value if len(eval_fw.results) > 7 else 0,   # summary quality
    eval_fw.results[8].value if len(eval_fw.results) > 8 else 0,   # cost efficiency
]
colors = ["tab:blue" if v >= 0.7 else ("tab:orange" if v >= 0.5 else "tab:red") for v in values]
ax2.bar(metrics, values, color=colors, alpha=0.8)
ax2.set_ylim(0, 1.0)
ax2.set_ylabel("Score (0-1)")
ax2.set_title("Platform Evaluation Dashboard")
ax2.axhline(y=0.7, color="green", linestyle="--", alpha=0.5, label="target (0.7)")
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\n✓ Cost and throughput evaluation complete")

In [ ]:
# --- Final evaluation report ---
print("FINAL EVALUATION REPORT")
print("=" * 50)
report_df = eval_fw.report()
print(report_df[["stage", "metric", "value"]].to_string(index=False))
avg_score = report_df["value"].mean()
print(f"\nOverall average score: {avg_score:.3f}")
print(f"Stages evaluated: {report_df['stage'].nunique()}")
verdict = "READY" if avg_score > 0.6 else "NEEDS WORK"
print(f"\n✓ Evaluation complete — platform {verdict} for deployment")

## Exercises

1. **Expand the gold set** — Add 20 more labelled reviews covering edge cases:
   sarcasm ("Love waiting 3 days for support, really great experience"), mixed
   multi-aspect reviews, and very short reviews (< 10 words). Re-run the
   extraction evaluation and report the F1 change.

2. **LLM-as-judge calibration** — Have 3 human raters score the same 5
   summaries on the same 4 criteria. Compare human scores vs LLM-as-judge
   scores. Compute inter-rater agreement (Cohen's kappa) between LLM and each
   human rater.

3. **Cost sensitivity analysis** — Model the cost impact of switching from
   GPT-4o-mini to GPT-4o (10× token cost) and to a fine-tuned GPT-3.5
   (0.3× token cost). At what monthly volume does the fine-tuned model
   become worth the upfront fine-tuning investment ($500)?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Gold-labelled test sets are essential — 30 reviews minimum for meaningful P/R/F1 |
| 2 | Aspect extraction evaluation must track missed, hallucinated, and miscategorised aspects separately |
| 3 | Sentiment confusion matrices reveal systematic biases (e.g., mixed reviews classified as neutral) |
| 4 | Trend detection has a precision-recall tradeoff controlled by the z-score threshold |
| 5 | Time-to-detection measures real-world urgency — catching issues 1 day earlier prevents escalation |
| 6 | LLM-as-judge with structured criteria scales summary evaluation beyond human reviewer capacity |
| 7 | LLM pipelines are 99%+ cheaper than manual analysis at scale, with sub-second latency |

## What's Next

You've built and evaluated a complete feedback intelligence platform. Next
steps to production:
- Connect real data sources (Zendesk, App Store, social APIs)
- Fine-tune extraction on your domain's aspect taxonomy
- Build a real-time dashboard with Streamlit or Grafana
- Implement A/B testing on alert thresholds

Continue to **Lab 10 — Code Review & Security** for the next applied lab.

---

## References

1. Pontiki, M. et al., *SemEval-2016 Task 5: Aspect-Based Sentiment Analysis*, 2016 — <https://aclanthology.org/S16-1002/>
2. Zheng, L. et al., *Judging LLM-as-a-Judge*, NeurIPS 2023 — <https://arxiv.org/abs/2306.05685>
3. OpenAI, *Pricing — GPT-4o-mini*, 2024 — <https://openai.com/pricing>
4. Medallia, *Experience Management Platform* — <https://www.medallia.com/>